<a href="https://colab.research.google.com/github/KevinKenya/nairobi-connector-open-source/blob/main/nairobi-benchmarks/nairobiOsBenchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 Nairobi OS vs Pandas: NBA Dataset Benchmarks

This notebook benchmarks **Nairobi OS** (fused analytics engine) against **Pandas** using the real NBA Player Statistics dataset. All comparisons use identical operations for a fair head-to-head test.

**Workloads tested:**
- Statistical Distillation (mean, std_dev, skewness, kurtosis)
- Pearson Correlation between key player stats
- End-to-end pipeline (ingest + crunch + correlate)


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
eoinamoore_historical_nba_data_and_player_box_scores_path = kagglehub.dataset_download('eoinamoore/historical-nba-data-and-player-box-scores')

print('Data source import complete.')

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "/kaggle/input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install nairobi_os -q

## 🔧 Setup: Load the NBA Dataset & Start Engines

In [ ]:
import os
import subprocess
import stat
from pathlib import Path
import nairobi_os

# Use the path from kagglehub if available, otherwise look in standard Kaggle mount points
try:
    nba_input_dir = eoinamoore_historical_nba_data_and_player_box_scores_path
except NameError:
    # Fallback to common Kaggle mount paths
    possible_paths = [
        "/kaggle/input/historical-nba-data-and-player-box-scores",
        "/kaggle/input/eoinamoore/historical-nba-data-and-player-box-scores"
    ]
    nba_input_dir = next((p for p in possible_paths if os.path.exists(p)), possible_paths[0])

csv_files = []
for dirname, _, filenames in os.walk(nba_input_dir):
    for f in filenames:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(dirname, f))

if not csv_files:
    raise ValueError(f"No CSV files found in {nba_input_dir}. Please check your Kaggle data sources.")

print(f"Found {len(csv_files)} CSV files in {nba_input_dir}:")
for f in csv_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {f} ({size_mb:.1f} MB)")

# Use the largest CSV (player box scores) as our benchmark dataset
nba_dataset = max(csv_files, key=lambda f: os.path.getsize(f))
print(f"\n✅ Selected dataset: {nba_dataset}")

# Copy to working directory for consistent access
dataset_basename = os.path.basename(nba_dataset)
local_dataset = f"/kaggle/working/{dataset_basename}"
# Create working dir if it doesn't exist (for non-Kaggle envs)
os.makedirs("/kaggle/working", exist_ok=True)
!cp {nba_dataset} {local_dataset}

# Quick peek at the data
df_peek = pd.read_csv(local_dataset, low_memory=False)
print(f"\n📊 Dataset shape: {df_peek.shape}")
print(f"   Columns: {list(df_peek.columns[:15])}...")
print(f"   Sample rows:")
print(df_peek.head(3).to_string())

In [ ]:
# Start the Nairobi OS Axum Refinery daemon
import subprocess
from pathlib import Path

print("🛠 Installing D-Bus Infrastructure...")
subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "dbus-x11"], capture_output=True)

print("🔌 Initializing D-Bus Session...")
dbus_out = subprocess.check_output(["dbus-launch"]).decode()
for line in dbus_out.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        os.environ[k] = v.replace(";", "").replace("'", "").replace('"', '')

print("🔐 Granting Executable Permissions...")
bin_path = Path(nairobi_os.__file__).parent / "bin" / "nairobi-axum-refinery"
os.chmod(bin_path, 0o755)

print("🔥 Igniting the Heavy Iron (Nairobi OS)...")
try:
    nairobi_os.start_refinery()
    print("✅ EMPIRE ONLINE")
except Exception as e:
    print(f"\n💥 Ignition Failed: {e}")
    os.system("cat ~/.nairobi_refinery.log 2>/dev/null || echo 'No log file'")

## 📊 Benchmark 1: Statistical Distillation (10 Iterations)

Compute **mean, std_dev, skewness, kurtosis** on the `points` column — the core statistical primitives.

In [ ]:
import time
import json
import numpy as np
import pandas as pd
import nairobi_os

NUM_ITERATIONS = 10
column = "points"

# ── PANDA BENCHMARK ──────────────────────────────
print("🐼 Pandas Statistical Distillation...")
pandas_ingest_times = []
pandas_crunch_times = []
pandas_totals = []
pandas_results = None

for i in range(NUM_ITERATIONS):
    t0 = time.perf_counter_ns()
    df_p = pd.read_csv(local_dataset, low_memory=False)
    ingest_ms = (time.perf_counter_ns() - t0) / 1_000_000
    pandas_ingest_times.append(ingest_ms)

    t1 = time.perf_counter_ns()
    mean_val = df_p[column].mean()
    std_val = df_p[column].std()
    skew_val = df_p[column].skew()
    kurt_val = df_p[column].kurt()
    crunch_ms = (time.perf_counter_ns() - t1) / 1_000_000
    pandas_crunch_times.append(crunch_ms)
    pandas_totals.append(ingest_ms + crunch_ms)

    if pandas_results is None:
        pandas_results = {"mean": mean_val, "std_dev": std_val, "skewness": skew_val, "kurtosis": kurt_val}

print(f"  Mean ingest: {np.mean(pandas_ingest_times):.2f} ms")
print(f"  Mean crunch: {np.mean(pandas_crunch_times):.2f} ms")
print(f"  Mean total:  {np.mean(pandas_totals):.2f} ms")
print(f"  Results: {pandas_results}")

# ── NAIROBI BENCHMARK ───────────────────────────
print("\n🦁 Nairobi OS Statistical Distillation...")
nairobi_ingest_times = []
nairobi_crunch_times = []
nairobi_totals = []
nairobi_results = None

for i in range(NUM_ITERATIONS):
    t0 = time.perf_counter_ns()
    handle = nairobi_os.data.ingest(local_dataset)
    ingest_ms = (time.perf_counter_ns() - t0) / 1_000_000
    nairobi_ingest_times.append(ingest_ms)

    t1 = time.perf_counter_ns()
    crunch_json = nairobi_os.data.crunch(handle, column)
    crunch_ms = (time.perf_counter_ns() - t1) / 1_000_000
    nairobi_crunch_times.append(crunch_ms)
    nairobi_totals.append(ingest_ms + crunch_ms)

    if nairobi_results is None:
        crunch_res = json.loads(crunch_json)
        nairobi_results = {
            "mean": crunch_res["mean"],
            "std_dev": crunch_res["std_dev"],
            "skewness": crunch_res["skewness"],
            "kurtosis": crunch_res["kurtosis"]
        }

print(f"  Mean ingest: {np.mean(nairobi_ingest_times):.2f} ms")
print(f"  Mean crunch: {np.mean(nairobi_crunch_times):.2f} ms")
print(f"  Mean total:  {np.mean(nairobi_totals):.2f} ms")
print(f"  Results: {nairobi_results}")

# ── VALIDATION ───────────────────────────────────
print("\n✅ Math Validation:")
for key in ["mean", "std_dev", "skewness", "kurtosis"]:
    p = pandas_results[key]
    n = nairobi_results[key]
    diff = abs(p - n)
    match = "✅" if diff < 1e-5 else "❌"
    print(f"  {match} {key}: pandas={p:.6f}, nairobi={n:.6f}, diff={diff:.8f}")

# ── SUMMARY TABLE ────────────────────────────────
print("\n" + "="*60)
print("BENCHMARK SUMMARY: Statistical Distillation")
print("="*60)
print(f"{'Metric':<25} {'Pandas (ms)':<15} {'Nairobi (ms)':<15} {'Speedup':<10}")
print("-" * 60)
print(f"{'Ingest':<25} {np.mean(pandas_ingest_times):<15.2f} {np.mean(nairobi_ingest_times):<15.2f} {np.mean(pandas_ingest_times)/np.mean(nairobi_ingest_times):.2f}x")
print(f"{'Crunch':<25} {np.mean(pandas_crunch_times):<15.2f} {np.mean(nairobi_crunch_times):<15.2f} {np.mean(pandas_crunch_times)/np.mean(nairobi_crunch_times):.2f}x")
print(f"{'Total':<25} {np.mean(pandas_totals):<15.2f} {np.mean(nairobi_totals):<15.2f} {np.mean(pandas_totals)/np.mean(nairobi_totals):.2f}x")
print("="*60)

## 📊 Benchmark 2: Pearson Correlation (10 Iterations)

Compute Pearson correlation between **points** and **assists** — a pairwise statistical operation.

In [ ]:
# ── PANDA BENCHMARK ──────────────────────────────
print("🐼 Pandas Pearson Correlation...")
pandas_corr_times = []
pandas_corr_result = None

for i in range(NUM_ITERATIONS):
    df_p = pd.read_csv(local_dataset, low_memory=False)
    t0 = time.perf_counter_ns()
    corr_val = df_p["points"].corr(df_p["assists"], method="pearson")
    elapsed_ms = (time.perf_counter_ns() - t0) / 1_000_000
    pandas_corr_times.append(elapsed_ms)
    if pandas_corr_result is None:
        pandas_corr_result = corr_val

print(f"  Mean time: {np.mean(pandas_corr_times):.2f} ms")
print(f"  Result: {pandas_corr_result:.6f}")

# ── NAIROBI BENCHMARK ───────────────────────────
print("\n🦁 Nairobi OS Pearson Correlation...")
nairobi_corr_times = []
nairobi_corr_result = None

for i in range(NUM_ITERATIONS):
    handle = nairobi_os.data.ingest(local_dataset)
    t0 = time.perf_counter_ns()
    corr_json = nairobi_os.data.correlate(handle, "points,assists")
    elapsed_ms = (time.perf_counter_ns() - t0) / 1_000_000
    nairobi_corr_times.append(elapsed_ms)
    if nairobi_corr_result is None:
        corr_res = json.loads(corr_json)
        nairobi_corr_result = corr_res["pearson"]

print(f"  Mean time: {np.mean(nairobi_corr_times):.2f} ms")
print(f"  Result: {nairobi_corr_result:.6f}")

# ── VALIDATION ───────────────────────────────────
print(f"\n✅ Pearson Validation: diff = {abs(pandas_corr_result - nairobi_corr_result):.8f}")

# ── SUMMARY ──────────────────────────────────────
print("\n" + "="*60)
print("BENCHMARK SUMMARY: Pearson Correlation")
print("="*60)
print(f"{'Metric':<25} {'Pandas (ms)':<15} {'Nairobi (ms)':<15} {'Speedup':<10}")
print("-" * 60)
print(f"{'Correlation':<25} {np.mean(pandas_corr_times):<15.2f} {np.mean(nairobi_corr_times):<15.2f} {np.mean(pandas_corr_times)/np.mean(nairobi_corr_times):.2f}x")
print("="*60)

## 📊 Benchmark 3: Fused Pipeline (Ingest + Crunch + Correlate)

End-to-end fused analytics: ingest, compute statistics, and correlate in a single call via `nairobi_os.data.pipeline()`.

In [ ]:
# ── FUSED PIPELINE BENCHMARK ─────────────────────
print("🦁 Nairobi OS Fused Pipeline (Ingest + Crunch + Correlate)...")
pipeline_times = []
pipeline_result = None

for i in range(NUM_ITERATIONS):
    t0 = time.perf_counter_ns()
    result_json = nairobi_os.data.pipeline(
        local_dataset,
        "points",
        "assists,rebounds"
    )
    elapsed_ms = (time.perf_counter_ns() - t0) / 1_000_000
    pipeline_times.append(elapsed_ms)
    if pipeline_result is None:
        pipeline_result = json.loads(result_json)

print(f"  Mean total time: {np.mean(pipeline_times):.2f} ms")
print(f"  Result keys: {list(pipeline_result.keys())}")

# ── COMPARISON: Pandas equivalent (3 separate ops) ──
print("\n🐼 Pandas Equivalent (Read + Stats + Correlate)...")
pandas_pipeline_times = []

for i in range(NUM_ITERATIONS):
    t0 = time.perf_counter_ns()
    df_p = pd.read_csv(local_dataset, low_memory=False)
    _ = df_p["points"].mean()
    _ = df_p["points"].std()
    _ = df_p["assists"].corr(df_p["rebounds"], method="pearson")
    elapsed_ms = (time.perf_counter_ns() - t0) / 1_000_000
    pandas_pipeline_times.append(elapsed_ms)

print(f"  Mean total time: {np.mean(pandas_pipeline_times):.2f} ms")

# ── SUMMARY ──────────────────────────────────────
print("\n" + "="*60)
print("BENCHMARK SUMMARY: Fused Pipeline vs Pandas")
print("="*60)
pipeline_speedup = np.mean(pandas_pipeline_times) / np.mean(pipeline_times)
print(f"{'Metric':<30} {'Pandas (ms)':<15} {'Nairobi (ms)':<15} {'Speedup':<10}")
print("-" * 60)
print(f"{'Full Pipeline':<30} {np.mean(pandas_pipeline_times):<15.2f} {np.mean(pipeline_times):<15.2f} {pipeline_speedup:.2f}x")
print("="*60)

## 📈 Results Summary

Run all three benchmarks above and compare the results table below.

In [ ]:
import pandas as pd
import numpy as np

# Collect results from all benchmarks
results = {
    "Benchmark": [
        "Statistical Distillation (Ingest)",
        "Statistical Distillation (Crunch)",
        "Statistical Distillation (Total)",
        "Pearson Correlation",
        "Fused Pipeline (Nairobi)",
        "Fused Pipeline (Pandas)",
    ],
    "Pandas (ms)": [
        np.mean(pandas_ingest_times),
        np.mean(pandas_crunch_times),
        np.mean(pandas_totals),
        np.mean(pandas_corr_times),
        np.mean(pandas_pipeline_times),
        np.mean(pandas_pipeline_times),
    ],
    "Nairobi (ms)": [
        np.mean(nairobi_ingest_times),
        np.mean(nairobi_crunch_times),
        np.mean(nairobi_totals),
        np.mean(nairobi_corr_times),
        np.mean(pipeline_times),
        np.mean(pandas_pipeline_times),
    ],
}

df_results = pd.DataFrame(results)
df_results["Speedup"] = df_results["Pandas (ms)"] / df_results["Nairobi (ms)"]
df_results["Pandas (ms)"] = df_results["Pandas (ms)"].apply(lambda x: f"{x:.2f}")
df_results["Nairobi (ms)"] = df_results["Nairobi (ms)"].apply(lambda x: f"{x:.2f}")
df_results["Speedup"] = df_results["Speedup"].apply(lambda x: f"{x:.2f}x")

from IPython.display import display, HTML
display(HTML('<h3>🏆 Nairobi OS vs Pandas — Benchmark Results</h3>'))
display(df_results.to_html(index=False, table_id='bench-table'))

## 🔧 Teardown

Stop the Nairobi OS refinery daemon when done.

In [ ]:
nairobi_os.stop_refinery()
print("🛑 Nairobi OS refinery stopped.")